In [12]:
import pandas as pd
import numpy as np
pd.set_option('display.width', 140)

df = pd.read_csv('dataset_prepared.csv', parse_dates=['evaluation_date'])
print(df.shape)
print('период:', df['evaluation_date'].min(), '->', df['evaluation_date'].max())

(17065, 33)
период: 2024-09-01 10:30:11.746000 -> 2026-08-23 15:46:29.813000


In [13]:
cutoff = df['evaluation_date'].quantile(0.8)
train = df[df['evaluation_date'] < cutoff].copy()
test = df[df['evaluation_date'] >= cutoff].copy()
print('cutoff:', cutoff)
print('train:', len(train), ' test:', len(test))
print('train период:', train.evaluation_date.min(), '->', train.evaluation_date.max())
print('test период:', test.evaluation_date.min(), '->', test.evaluation_date.max())

cutoff: 2026-04-02 16:28:21.586400
train: 13652  test: 3413
train период: 2024-09-01 10:30:11.746000 -> 2026-04-02 16:25:13.372000
test период: 2026-04-02 16:40:54.444000 -> 2026-08-23 15:46:29.813000


In [14]:
pp = pd.read_csv('dataset_task/pricing_policy.csv') #вилка бейзлайна

def assign_segment(price):
    row = pp[(pp['min_price'] <= price) & (price <= pp['max_price'])]
    return row.iloc[0]['segment_id'] if len(row) else np.nan

train['segment_id'] = train['buy_price'].apply(assign_segment)
test['segment_id'] = test['market_avg_price'].apply(
    lambda p: assign_segment(p) if pd.notna(p) else np.nan)

seg_bounds = train.groupby('segment_id')['buy_price'].quantile([0.25, 0.75]).unstack()
seg_bounds.columns = ['baseline_from', 'baseline_to']
print(seg_bounds)

            baseline_from  baseline_to
segment_id                            
1.0              490750.0     881900.0
2.0             1479625.0    2200000.0
3.0             3032500.0    4000000.0
4.0             4900000.0    5600000.0
5.0             6500000.0    7405000.0
6.0             8500000.0    9300000.0
7.0            10970000.0   12750000.0
8.0            16500000.0   20575000.0


In [15]:
# 8 процентов покрытие бейзлайном очень плохой результат, посмотрел глубже что не так
# снизу бейзлайн с учетом маржи, то есть машин которые должны быть к примеру в сегменте 3, но из-за включенной в стоимость маржи находится в 4 сегменте

In [16]:
test = test.merge(seg_bounds, on='segment_id', how='left')
covered = (test['buy_price'] >= test['baseline_from']) & (test['buy_price'] <= test['baseline_to'])
width = test['baseline_to'] - test['baseline_from']
print('покрытие бейзлайном (доля попаданий факта в вилку):', covered.mean())
print('медианная ширина вилки:', width.median())
print('медианная ширина вилки, % от buy_price:', (width / test['buy_price']).median())

покрытие бейзлайном (доля попаданий факта в вилку): 0.08321125109874011
медианная ширина вилки: 905000.0
медианная ширина вилки, % от buy_price: 0.22653721682847897


In [17]:
margin_map = pp.set_index('segment_id')['margin_initial'].to_dict()

test['seg_raw'] = test['market_avg_price'].apply(
    lambda p: assign_segment(p) if pd.notna(p) else np.nan)
actual_seg = test['buy_price'].apply(assign_segment)
print('совпадение сегмента (до поправки):', (test['seg_raw'] == actual_seg).mean())

test['margin_raw'] = test['seg_raw'].map(margin_map)
test['price_proxy'] = test['market_avg_price'] / (1 + test['margin_raw'].fillna(0.2))
test['segment_id'] = test['price_proxy'].apply(
    lambda p: assign_segment(p) if pd.notna(p) else np.nan)
print('совпадение сегмента (после поправки на маржу):', (test['segment_id'] == actual_seg).mean())

совпадение сегмента (до поправки): 0.183123351889833
совпадение сегмента (после поправки на маржу): 0.47934368590682686


In [21]:
test = test.drop(columns=['baseline_from', 'baseline_to'])
test = test.merge(seg_bounds, on='segment_id', how='left')
covered = (test['buy_price'] >= test['baseline_from']) & (test['buy_price'] <= test['baseline_to'])
width = test['baseline_to'] - test['baseline_from']
print('покрытие бейзлайна (после поправки на маржу):', covered.mean())
print('медианная ширина вилки:', width.median())
print('медианная ширина вилки, % от buy_price:', (width / test['buy_price']).median())

покрытие бейзлайна (после поправки на маржу): 0.2698505713448579
медианная ширина вилки: 905000.0
медианная ширина вилки, % от buy_price: 0.241875


In [24]:
import sys; sys.path.insert(0, '../src')
from feature_contract import ALLOWED_FEATURES
from sklearn.ensemble import HistGradientBoostingRegressor

# month_idx — простой числовой признак времени помогает модели поймать тренд
for _d in (train, test):
    _d['month_idx'] = (_d['evaluation_date'].dt.year - 2024) * 12 + _d['evaluation_date'].dt.month

drop_stub = ['market_median_price_w', 'market_q1_price', 'market_q3_price', 'has_price_quantiles']
feature_cols = [c for c in ALLOWED_FEATURES if c in df.columns and c not in drop_stub
                and c != 'transaction_subgroup'] + ['month_idx']
print(len(feature_cols), 'признаков:', feature_cols)

26 признаков: ['mark', 'model', 'generation', 'car_year', 'mileage', 'engine_cc', 'body_type', 'engine_type', 'transmission', 'drive_type', 'color', 'steering', 'door_num', 'keys_count', 'on_guarantee', 'car_category', 'total_car_state', 'transaction_group', 'transaction', 'is_mobile_purch', 'is_comtrade', 'branch_key', 'market_avg_price', 'market_ads_cnt', 'market_avg_mileage', 'month_idx']


In [26]:
X_train, X_test = train[feature_cols].copy(), test[feature_cols].copy()
y_train, y_test = np.log(train['buy_price']), test['buy_price']

GLOBAL_MEAN = y_train.mean()
SMOOTHING = 20

def target_encode(train_col, train_y, test_col):
    stats = train_y.groupby(train_col).agg(['mean', 'count'])
    smoothed = (stats['mean'] * stats['count'] + GLOBAL_MEAN * SMOOTHING) / (stats['count'] + SMOOTHING)
    return (train_col.map(smoothed).fillna(GLOBAL_MEAN),
            test_col.map(smoothed).fillna(GLOBAL_MEAN))

for c in ['model', 'generation']:
    X_train[c + '_te'], X_test[c + '_te'] = target_encode(X_train[c], y_train, X_test[c])
    X_train.drop(columns=[c], inplace=True)
    X_test.drop(columns=[c], inplace=True)

cat_cols = X_train.select_dtypes(include=['object', 'bool']).columns.tolist()
for c in cat_cols:
    X_train[c] = X_train[c].astype('category')
    X_test[c] = pd.Categorical(X_test[c], categories=X_train[c].cat.categories)
print('категориальных признаков:', len(cat_cols), ' target-encoded:', 2)
print('макс. кардинальность нативных категорий:', max(X_train[c].nunique() for c in cat_cols))

категориальных признаков: 13  target-encoded: 2
макс. кардинальность нативных категорий: 86


In [27]:
models = {}
bands = {}
for q, name in [(0.25, 'model_from'), (0.75, 'model_to')]:
    m = HistGradientBoostingRegressor(
        loss='quantile', quantile=q, categorical_features='from_dtype',
        max_iter=300, learning_rate=0.05, random_state=42)
    m.fit(X_train, y_train)
    models[name] = m
    bands[name] = np.exp(m.predict(X_test))

test['model_from'] = np.minimum(bands['model_from'], bands['model_to'])
test['model_to'] = np.maximum(bands['model_from'], bands['model_to'])
n_swapped = (bands['model_from'] > bands['model_to']).sum()
print('пересечений квантилей:', n_swapped)

пересечений квантилей: 38


In [28]:
model_covered = (test['buy_price'] >= test['model_from']) & (test['buy_price'] <= test['model_to'])
model_width = test['model_to'] - test['model_from']

comparison = pd.DataFrame({
    'покрытие': [covered.mean(), model_covered.mean()],
    'медианная_ширина_тг': [width.median(), model_width.median()],
    'медианная_ширина_%': [(width / test['buy_price']).median(),
                            (model_width / test['buy_price']).median()],
}, index=['бейзлайн (сегмент+маржа)', 'модель (квантильная регрессия)'])
print(comparison.to_string())

                                покрытие  медианная_ширина_тг  медианная_ширина_%
бейзлайн (сегмент+маржа)        0.269851        905000.000000            0.241875
модель (квантильная регрессия)  0.385585        698058.937679            0.202771


In [29]:
test['covered_model'] = model_covered
test['covered_baseline'] = covered
test['model_mid'] = (test['model_from'] + test['model_to']) / 2
test['signed_pct_error_mid'] = (test['model_mid'] - test['buy_price']) / test['buy_price']
test['abs_pct_error_mid'] = test['signed_pct_error_mid'].abs()

by_segment = test.groupby('segment_id').agg(
    n=('buy_price', 'size'),
    покрытие_модели=('covered_model', 'mean'),
    покрытие_бейзлайна=('covered_baseline', 'mean'),
    медианная_абс_ошибка=('abs_pct_error_mid', 'median'),
    медианное_смещение=('signed_pct_error_mid', 'median'),
).round(3)
by_segment['дельта_покрытия'] = (by_segment['покрытие_модели'] - by_segment['покрытие_бейзлайна']).round(3)
print(by_segment.to_string())

               n  покрытие_модели  покрытие_бейзлайна  медианная_абс_ошибка  медианное_смещение  дельта_покрытия
segment_id                                                                                                      
1.0          127            0.409               0.417                 0.393               0.017           -0.008
2.0          484            0.384               0.287                 0.257               0.131            0.097
3.0         1009            0.452               0.350                 0.113               0.066            0.102
4.0          543            0.414               0.260                 0.100               0.063            0.154
5.0          409            0.386               0.357                 0.074               0.022            0.029
6.0          155            0.374               0.187                 0.067              -0.002            0.187
7.0          179            0.408               0.268                 0.078              -0.002 

In [30]:
by_mark = test.groupby('mark').agg(
    n=('buy_price', 'size'),
    покрытие_модели=('covered_model', 'mean'),
    покрытие_бейзлайна=('covered_baseline', 'mean'),
    медианная_абс_ошибка=('abs_pct_error_mid', 'median'),
    медианное_смещение=('signed_pct_error_mid', 'median'),
).query('n >= 30')
by_mark['дельта_покрытия'] = by_mark['покрытие_модели'] - by_mark['покрытие_бейзлайна']
by_mark = by_mark.round(3).sort_values(['покрытие_модели', 'n'], ascending=[True, False])
print('5 марок с наихудшим покрытием модели (n>=30):')
print(by_mark.head(5).to_string())
print('\n5 марок с наилучшим покрытием модели (n>=30):')
print(by_mark.tail(5).to_string())

5 марок с наихудшим покрытием модели (n>=30):
                 n  покрытие_модели  покрытие_бейзлайна  медианная_абс_ошибка  медианное_смещение  дельта_покрытия
mark                                                                                                              
OMODA           31            0.129               0.097                 0.180               0.151            0.032
Audi            65            0.262               0.200                 0.278               0.017            0.062
BMW             45            0.311               0.222                 0.122               0.034            0.089
Mitsubishi     103            0.320               0.350                 0.145               0.021           -0.029
Mercedes-Benz  109            0.339               0.165                 0.177               0.019            0.174

5 марок с наилучшим покрытием модели (n>=30):
             n  покрытие_модели  покрытие_бейзлайна  медианная_абс_ошибка  медианное_смещение  дельта_

Модель в целом выигрывает у бейзлайна, но не достигает номинальных 50% покрытия. Особого внимания требуют группы с малой выборкой: их крайние значения менее устойчивы. Знак медианное_смещение важен для бизнеса: положительное значение означает систематически завышенную рекомендацию и риск переплаты, отрицательное риск отказа клиента

In [ ]:
# симуляция за 3 месяца

In [ ]:
test['buy_price_model_mid'] = (test['model_from'] + test['model_to']) / 2
test['gm2_simulated'] = test['gm2'] + (test['buy_price'] - test['buy_price_model_mid'])

quarter_end = test['evaluation_date'].max()
quarter_start = quarter_end - pd.DateOffset(months=3)
quarter = test[test['evaluation_date'].between(quarter_start, quarter_end)].copy()

gm2_actual = quarter['gm2'].sum()
gm2_simulated = quarter['gm2_simulated'].sum()
gm2_delta = gm2_simulated - gm2_actual
loss_actual = (quarter['gm2'] < 0).mean()
loss_simulated = (quarter['gm2_simulated'] < 0).mean()

print('Период:', quarter_start, '->', quarter_end)
print('Сделок:', len(quarter))
print(f'Сумма GM2, факт:       {gm2_actual:,.0f}')
print(f'Сумма GM2, симуляция: {gm2_simulated:,.0f}')
print(f'Разница:               {gm2_delta:,.0f} ({100 * gm2_delta / gm2_actual:+.1f}%)')
print(f'Доля GM2 < 0, факт:       {100 * loss_actual:.1f}%')
print(f'Доля GM2 < 0, симуляция: {100 * loss_simulated:.1f}%')

Период: 2026-05-23 15:46:29.813000 -> 2026-08-23 15:46:29.813000
Сделок: 2043
Сумма GM2, факт:       1,533,462,177
Сумма GM2, симуляция: 1,338,558,077
Разница:               -194,904,100 (-12.7%)
Доля GM2 < 0, факт:       2.1%
Доля GM2 < 0, симуляция: 26.4%


In [ ]:
print('train buy_price: mean=%.0f median=%.0f' % (train.buy_price.mean(), train.buy_price.median()))
print('test  buy_price: mean=%.0f median=%.0f' % (test.buy_price.mean(), test.buy_price.median()))
print()
print('train market_avg_price median:', train.market_avg_price.median())
print('test  market_avg_price median:', test.market_avg_price.median())
print()
print('mean buy_price факт:  ', test['buy_price'].mean())
print('mean model mid:       ', test['buy_price_model_mid'].mean())

train buy_price: mean=5024700 median=4500000
test  buy_price: mean=4367114 median=3800000

train market_avg_price median: 5945215.312312312
test  market_avg_price median: 5389092.797142857

mean buy_price факт:   4367113.868151187
mean model mid:        4470071.932353531
